In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

BASE_DIR = "/content/drive/MyDrive/SafeCue Audio Recordings"

print("Folder exists:", os.path.exists(BASE_DIR))
print("\nContents:")

for item in os.listdir(BASE_DIR):
    print("📁", item)

Folder exists: True

Contents:
📁 Siren
📁 Fire Alarm
📁 Smoke Alarm
📁 Screaming
📁 Background Noise


In [4]:
!pip install -q yt-dlp
!apt-get -qq update
!apt-get -qq install ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 30.3 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [5]:
import os

METADATA_DIR = os.path.join(BASE_DIR, "_metadata")
os.makedirs(METADATA_DIR, exist_ok=True)

print("Metadata folder:", METADATA_DIR)

Metadata folder: /content/drive/MyDrive/SafeCue Audio Recordings/_metadata


In [6]:
import pandas as pd

metadata_url = "https://research.google.com/audioset/download.html"

print("AudioSet metadata page:")
print(metadata_url)

AudioSet metadata page:
https://research.google.com/audioset/download.html


In [7]:
!wget -q --show-progress \
"https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/balanced_train_segments.csv" \
-o /dev/null

In [8]:
import os

print(os.listdir(METADATA_DIR))

[]


In [9]:
import requests

url = "https://research.google.com/audioset/download.html"
response = requests.get(url)

print("Status code:", response.status_code)
print("Page downloaded:", len(response.text) > 0)

Status code: 200
Page downloaded: True


In [10]:
import requests

url = "http://storage.googleapis.com/us_audioset/youtube_corpus/v1/csv/balanced_train_segments.csv"

output_path = os.path.join(METADATA_DIR, "balanced_train_segments.csv")

response = requests.get(url)

with open(output_path, "wb") as f:
    f.write(response.content)

print("Downloaded:", output_path)
print("File size:", os.path.getsize(output_path), "bytes")

Downloaded: /content/drive/MyDrive/SafeCue Audio Recordings/_metadata/balanced_train_segments.csv
File size: 1211931 bytes


In [14]:
csv_path = os.path.join(METADATA_DIR, "balanced_train_segments.csv")

with open(csv_path, "r", encoding="utf-8") as f:
    for i in range(8):
        print(f"LINE {i+1}:", repr(f.readline()))

LINE 1: '# Segments csv created Sun Mar  5 10:54:31 2017\n'
LINE 2: '# num_ytids=22160, num_segs=22160, num_unique_labels=527, num_positive_labels=52882\n'
LINE 3: '# YTID, start_seconds, end_seconds, positive_labels\n'
LINE 4: '--PJHxphWEs, 30.000, 40.000, "/m/09x0r,/t/dd00088"\n'
LINE 5: '--ZhevVpy1s, 50.000, 60.000, "/m/012xff"\n'
LINE 6: '--aE2O5G5WE, 0.000, 10.000, "/m/03fwl,/m/04rlf,/m/09x0r"\n'
LINE 7: '--aO5cdqSAg, 30.000, 40.000, "/t/dd00003,/t/dd00005"\n'
LINE 8: '--aaILOrkII, 200.000, 210.000, "/m/032s66,/m/073cg4"\n'


In [15]:
import csv
import pandas as pd
import os

csv_path = os.path.join(METADATA_DIR, "balanced_train_segments.csv")

rows = []

with open(csv_path, "r", encoding="utf-8") as f:
    reader = csv.reader(f, skipinitialspace=True)

    for row in reader:
        if row and not row[0].startswith("#"):
            rows.append(row)

df = pd.DataFrame(
    rows,
    columns=["YTID", "start_seconds", "end_seconds", "positive_labels"]
)

print("Shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

Shape: (22160, 4)

First 5 rows:


,YTID,start_seconds,end_seconds,positive_labels
0,--PJHxphWEs,30.000,40.000,"/m/09x0r,/t/dd00088"
1,--ZhevVpy1s,50.000,60.000,/m/012xff
2,--aE2O5G5WE,0.000,10.000,"/m/03fwl,/m/04rlf,/m/09x0r"
3,--aO5cdqSAg,30.000,40.000,"/t/dd00003,/t/dd00005"
4,--aaILOrkII,200.000,210.000,"/m/032s66,/m/073cg4"


In [16]:
import requests
import json
import os

ontology_url = "https://raw.githubusercontent.com/audioset/ontology/master/ontology.json"
ontology_path = os.path.join(METADATA_DIR, "ontology.json")

response = requests.get(ontology_url)
response.raise_for_status()

with open(ontology_path, "wb") as f:
    f.write(response.content)

print("Ontology downloaded successfully!")
print("File size:", os.path.getsize(ontology_path), "bytes")

Ontology downloaded successfully!
File size: 342780 bytes


In [17]:
import json

with open(ontology_path, "r", encoding="utf-8") as f:
    ontology = json.load(f)

keywords = ["siren", "fire alarm", "smoke alarm", "screaming", "background noise"]

for keyword in keywords:
    print(f"\n=== {keyword.upper()} ===")

    matches = []
    for item in ontology:
        name = item.get("name", "")
        if keyword.lower() in name.lower():
            matches.append((item.get("id"), name))

    for audio_id, name in matches:
        print(audio_id, "→", name)


=== SIREN ===
/m/04qvtq → Police car (siren)
/m/012n7d → Ambulance (siren)
/m/012ndj → Fire engine, fire truck (siren)
/m/03kmc9 → Siren
/m/0dgbq → Civil defense siren

=== FIRE ALARM ===
/m/0c3f7m → Fire alarm

=== SMOKE ALARM ===
/m/01y3hg → Smoke detector, smoke alarm

=== SCREAMING ===
/m/03qc9zr → Screaming

=== BACKGROUND NOISE ===
/m/093_4n → Background noise


In [18]:
label_ids = {
    "Siren": "/m/03kmc9",
    "Fire Alarm": "/m/0c3f7m",
    "Smoke Alarm": "/m/01y3hg",
    "Screaming": "/m/03qc9zr",
    "Background Noise": "/m/093_4n"
}

for category, label_id in label_ids.items():
    count = df["positive_labels"].apply(
        lambda x: label_id in x.split(",")
    ).sum()

    print(f"{category}: {count} clips")

Siren: 207 clips
Fire Alarm: 60 clips
Smoke Alarm: 60 clips
Screaming: 59 clips
Background Noise: 0 clips


In [19]:
import requests
import os

url = "http://storage.googleapis.com/us_audioset/youtube_corpus/v1/csv/unbalanced_train_segments.csv"

output_path = os.path.join(METADATA_DIR, "unbalanced_train_segments.csv")

response = requests.get(url)
response.raise_for_status()

with open(output_path, "wb") as f:
    f.write(response.content)

print("Downloaded successfully!")
print("File size:", os.path.getsize(output_path), "bytes")

Downloaded successfully!
File size: 101468408 bytes


In [20]:
import csv
from collections import Counter

unbalanced_path = os.path.join(
    METADATA_DIR,
    "unbalanced_train_segments.csv"
)

counts = Counter()

with open(unbalanced_path, "r", encoding="utf-8") as f:
    reader = csv.reader(f, skipinitialspace=True)

    for row in reader:
        if not row or row[0].startswith("#"):
            continue

        labels = row[3].split(",")

        for category, label_id in label_ids.items():
            if label_id in labels:
                counts[category] += 1

print("AudioSet unbalanced training counts:")
for category in label_ids:
    print(f"{category}: {counts[category]} clips")

AudioSet unbalanced training counts:
Siren: 8079 clips
Fire Alarm: 801 clips
Smoke Alarm: 426 clips
Screaming: 1126 clips
Background Noise: 0 clips


In [21]:
siren_id = "/m/03kmc9"

siren_rows = df[
    df["positive_labels"].apply(
        lambda x: siren_id in x.split(",")
    )
].reset_index(drop=True)

print("Siren clips available:", len(siren_rows))
print("\nFirst Siren clip:")
display(siren_rows.head(1))

Siren clips available: 207

First Siren clip:


,YTID,start_seconds,end_seconds,positive_labels
0,-4RWqM0UCCY,0.000,10.000,"/m/012n7d,/m/02mfyn,/m/03j1ly,/m/03kmc9"


In [23]:
import subprocess
import os
import time

# Number of clips to test
NUM_TEST = 10

# Use the first 10 Siren clips
test_rows = siren_rows.head(NUM_TEST)

siren_folder = os.path.join(BASE_DIR, "Siren")
os.makedirs(siren_folder, exist_ok=True)

success = 0
failed = 0

for i, row in test_rows.iterrows():
    ytid = row["YTID"]
    start = row["start_seconds"]
    end = row["end_seconds"]

    output_file = os.path.join(
        siren_folder,
        f"{ytid}_{start}_{end}.wav"
    )

    # Skip if already downloaded
    if os.path.exists(output_file):
        print(f"Already exists: {ytid}")
        success += 1
        continue

    url = f"https://www.youtube.com/watch?v={ytid}"

    command = [
        "yt-dlp",
        "--no-playlist",
        "--quiet",
        "--no-warnings",
        "--extract-audio",
        "--audio-format", "wav",
        "--download-sections", f"*{start}-{end}",
        "-o", output_file,
        url
    ]

    print(f"Downloading {success + failed + 1}/{NUM_TEST}: {ytid}")

    result = subprocess.run(
        command,
        capture_output=True,
        text=True
    )

    if result.returncode == 0 and os.path.exists(output_file):
        success += 1
        print("  ✅ Success")
    else:
        failed += 1
        print("  ❌ Failed")

    time.sleep(1)

print("\n====================")
print("DOWNLOAD TEST COMPLETE")
print("====================")
print("Successful:", success)
print("Failed:", failed)

Already exists: -4RWqM0UCCY
  ✅ Success
  ✅ Success
  ✅ Success
  ✅ Success
  ✅ Success
  ✅ Success
  ✅ Success
  ✅ Success
  ✅ Success

DOWNLOAD TEST COMPLETE
Successful: 10
Failed: 0


In [24]:
# Create a list of unique Siren YouTube sources
# We will use this list for the larger download.

siren_candidates = siren_rows.drop_duplicates(
    subset=["YTID"]
).reset_index(drop=True)

print("Total Siren candidates:", len(siren_candidates))
print("\nFirst 10 candidates:")
display(siren_candidates.head(10))

Total Siren candidates: 207

First 10 candidates:


,YTID,start_seconds,end_seconds,positive_labels
0,-4RWqM0UCCY,0.000,10.000,"/m/012n7d,/m/02mfyn,/m/03j1ly,/m/03kmc9"
1,-Y1qiiugnk8,30.000,40.000,"/m/012n7d,/m/03j1ly,/m/03kmc9"
2,-_dElQcyJnA,30.000,40.000,"/m/012n7d,/m/03j1ly,/m/03kmc9,/m/04qvtq"
3,-gKNRXbpAKs,30.000,40.000,"/m/012n7d,/m/03j1ly,/m/03kmc9,/m/0dgbq"
4,-oSw8wNpLcY,60.000,70.000,"/m/030rvx,/m/03kmc9"
5,-z47zN0YyE4,420.000,430.000,"/m/012ndj,/m/03j1ly,/m/03kmc9"
6,03NLMEMi8-I,30.000,40.000,"/m/03kmc9,/m/0dgbq"
7,0JKcTVpby0I,30.000,40.000,"/m/03kmc9,/m/0dgbq"
8,0JPT13OUVV8,220.000,230.000,"/m/012ndj,/m/03j1ly,/m/03kmc9"
9,0PhU-PIsUMw,80.000,90.000,"/m/03kmc9,/m/0dgbq"


In [25]:
import csv
import pandas as pd
import os

unbalanced_path = os.path.join(
    METADATA_DIR,
    "unbalanced_train_segments.csv"
)

siren_id = "/m/03kmc9"

siren_unbalanced_rows = []

with open(unbalanced_path, "r", encoding="utf-8") as f:
    reader = csv.reader(f, skipinitialspace=True)

    for row in reader:
        if not row or row[0].startswith("#"):
            continue

        labels = row[3].split(",")

        if siren_id in labels:
            siren_unbalanced_rows.append(row)

siren_unbalanced = pd.DataFrame(
    siren_unbalanced_rows,
    columns=["YTID", "start_seconds", "end_seconds", "positive_labels"]
)

# Remove duplicate YouTube videos
siren_unbalanced = siren_unbalanced.drop_duplicates(
    subset=["YTID"]
).reset_index(drop=True)

print("Unique Siren candidates in unbalanced set:", len(siren_unbalanced))
display(siren_unbalanced.head(10))

Unique Siren candidates in unbalanced set: 8079


,YTID,start_seconds,end_seconds,positive_labels
0,--7T50tAIrg,6.000,16.000,"/m/03kmc9,/m/09x0r"
1,-09rxiqNNEs,30.000,40.000,"/m/03kmc9,/m/0dgbq"
2,-0Eem_FuIto,15.000,25.000,"/m/012ndj,/m/03j1ly,/m/03kmc9"
3,-122tCXtFhU,30.000,40.000,"/m/03j1ly,/m/03kmc9,/m/04qvtq,/m/07yv9,/m/0k4j"
4,-2GlU3e0nTU,170.000,180.000,"/m/03j1ly,/m/03kmc9,/m/04qvtq"
5,-34Lznw_wUA,10.000,20.000,/m/03kmc9
6,-3qh-WFUV2U,30.000,40.000,"/m/03kmc9,/m/0dgbq"
7,-4JG_Ag99hY,30.000,40.000,"/m/03kmc9,/m/0dgbq"
8,-5EiLqHCLuI,0.000,10.000,"/m/012n7d,/m/03j1ly,/m/03kmc9"
9,-5o4C4ZtHSU,30.000,40.000,"/m/03j1ly,/m/03kmc9,/m/04qvtq"


In [26]:
# Select 260 Siren candidates for downloading
# We use a fixed random seed so the selection is reproducible.

SIREN_TARGET = 260

siren_download_candidates = siren_unbalanced.sample(
    n=SIREN_TARGET,
    random_state=42
).reset_index(drop=True)

print("Siren candidates selected:", len(siren_download_candidates))

display(siren_download_candidates.head(10))

Siren candidates selected: 260


,YTID,start_seconds,end_seconds,positive_labels
0,JngI6xTRE2o,30.000,40.000,"/m/012n7d,/m/03j1ly,/m/03kmc9,/m/04qvtq"
1,uaFJj5A9jCE,2.000,12.000,"/m/012ndj,/m/03j1ly,/m/03kmc9"
2,w8IVfFqEXjw,60.000,70.000,"/m/03kmc9,/m/0dgbq"
3,ysJhEu_L7lc,30.000,40.000,/m/03kmc9
4,r-ww2qIp1BE,30.000,40.000,"/m/03j1ly,/m/03kmc9,/m/04qvtq"
5,NHcHQ6ptBmA,30.000,40.000,/m/03kmc9
6,ctyG4PM_rtw,0.000,10.000,"/m/0342h,/m/03kmc9,/m/04szw,/m/0fx80y"
7,azXoLk-FCrU,30.000,40.000,"/m/03kmc9,/m/04qvtq,/m/07yv9"
8,CiYGv42LsZ8,28.000,38.000,"/m/03kmc9,/m/07yv9,/m/0k4j"
9,axOkcNpjyb4,30.000,40.000,"/m/03j1ly,/m/03kmc9,/m/04qvtq"


In [27]:
import subprocess
import os
import time
import pandas as pd

siren_folder = os.path.join(BASE_DIR, "Siren")
os.makedirs(siren_folder, exist_ok=True)

results = []

for i, row in siren_download_candidates.iterrows():

    ytid = row["YTID"]
    start = row["start_seconds"]
    end = row["end_seconds"]

    output_file = os.path.join(
        siren_folder,
        f"{ytid}_{start}_{end}.wav"
    )

    # If already downloaded, don't download again
    if os.path.exists(output_file):
        print(f"{i+1}/{len(siren_download_candidates)} ✅ Already exists: {ytid}")

        results.append({
            "YTID": ytid,
            "start_seconds": start,
            "end_seconds": end,
            "label": "Siren",
            "filename": output_file,
            "status": "already_exists"
        })
        continue

    url = f"https://www.youtube.com/watch?v={ytid}"

    command = [
        "yt-dlp",
        "--no-playlist",
        "--quiet",
        "--no-warnings",
        "--extract-audio",
        "--audio-format", "wav",
        "--download-sections", f"*{start}-{end}",
        "-o", output_file,
        url
    ]

    print(f"{i+1}/{len(siren_download_candidates)} Downloading: {ytid}")

    result = subprocess.run(
        command,
        capture_output=True,
        text=True
    )

    if result.returncode == 0 and os.path.exists(output_file):
        status = "success"
        print("   ✅ Success")
    else:
        status = "failed"
        print("   ❌ Failed")

    results.append({
        "YTID": ytid,
        "start_seconds": start,
        "end_seconds": end,
        "label": "Siren",
        "filename": output_file if status == "success" else "",
        "status": status
    })

    time.sleep(1)

# Save download results
download_results = pd.DataFrame(results)

results_path = os.path.join(
    METADATA_DIR,
    "siren_download_results.csv"
)

download_results.to_csv(results_path, index=False)

print("\n==============================")
print("SIREN DOWNLOAD COMPLETE")
print("==============================")
print("Successful:", (download_results["status"] == "success").sum())
print("Already existed:", (download_results["status"] == "already_exists").sum())
print("Failed:", (download_results["status"] == "failed").sum())
print("Results saved to:", results_path)

1/260 Downloading: JngI6xTRE2o
   ✅ Success
2/260 Downloading: uaFJj5A9jCE
   ✅ Success
3/260 Downloading: w8IVfFqEXjw
   ✅ Success
4/260 Downloading: ysJhEu_L7lc
   ✅ Success
5/260 Downloading: r-ww2qIp1BE
   ❌ Failed
6/260 Downloading: NHcHQ6ptBmA
   ✅ Success
7/260 Downloading: ctyG4PM_rtw
   ✅ Success
8/260 Downloading: azXoLk-FCrU
   ✅ Success
9/260 Downloading: CiYGv42LsZ8
   ✅ Success
10/260 Downloading: axOkcNpjyb4
   ✅ Success
11/260 Downloading: 1Q61oRUkfC0
   ✅ Success
12/260 Downloading: Lg6yI5szlVw
   ✅ Success
13/260 Downloading: yDeuZ92Y0eQ
   ✅ Success
14/260 Downloading: 2iKT4cvbJy8
   ❌ Failed
15/260 Downloading: 3LblLgNthRs
   ✅ Success
16/260 Downloading: FF410bQhJ8s
   ✅ Success
17/260 Downloading: HWeSf6Dsa5g
   ✅ Success
18/260 Downloading: 79-rErKgp3s
   ✅ Success
19/260 Downloading: vVEpMHmcH5Q
   ✅ Success
20/260 Downloading: eRZyNpT-XB8
   ✅ Success
21/260 Downloading: lAyZQxneqGA
   ✅ Success
22/260 Downloading: 0rdPzsSkzOU
   ✅ Success
23/260 Downloading: 0

In [28]:
# Get Fire Alarm candidates from AudioSet unbalanced metadata

fire_alarm_id = label_ids["Fire Alarm"]

fire_alarm_rows = []

with open(unbalanced_path, "r", encoding="utf-8") as f:
    reader = csv.reader(f, skipinitialspace=True)

    for row in reader:
        if not row or row[0].startswith("#"):
            continue

        labels = row[3].split(",")

        if fire_alarm_id in labels:
            fire_alarm_rows.append(row)

fire_alarm_candidates = pd.DataFrame(
    fire_alarm_rows,
    columns=["YTID", "start_seconds", "end_seconds", "positive_labels"]
)

fire_alarm_candidates = fire_alarm_candidates.drop_duplicates(
    subset=["YTID"]
).reset_index(drop=True)

print("Unique Fire Alarm candidates:", len(fire_alarm_candidates))
display(fire_alarm_candidates.head(10))

Unique Fire Alarm candidates: 801


,YTID,start_seconds,end_seconds,positive_labels
0,--OMDPXfO6o,9.000,19.000,"/m/04rlf,/m/09x0r,/m/0c3f7m"
1,-6KfoGO_o9M,220.000,230.000,"/m/09x0r,/m/0c3f7m"
2,-E9o4ywIoj8,10.000,20.000,/m/0c3f7m
3,-GSPHFcDEA0,190.000,200.000,"/m/09x0r,/m/0c3f7m"
4,-M88hG5X8fQ,10.000,20.000,/m/0c3f7m
5,-US7xnb1Lrw,0.000,10.000,"/m/09x0r,/m/0c3f7m"
6,-a9xYse4q-Y,70.000,80.000,"/m/030rvx,/m/0c3f7m"
7,-mDOld5RYxw,80.000,90.000,"/m/09x0r,/m/0c3f7m"
8,-ofHiiMgeuA,0.000,8.000,"/m/030rvx,/m/0c3f7m"
9,-syuIC_bDQQ,10.000,20.000,/m/0c3f7m


In [29]:
# Select Fire Alarm download candidates

FIRE_ALARM_TARGET = 260

fire_alarm_download_candidates = fire_alarm_candidates.sample(
    n=FIRE_ALARM_TARGET,
    random_state=42
).reset_index(drop=True)

print("Fire Alarm candidates selected:", len(fire_alarm_download_candidates))
display(fire_alarm_download_candidates.head(10))

Fire Alarm candidates selected: 260


,YTID,start_seconds,end_seconds,positive_labels
0,r0bhxdy3Ovk,30.000,40.000,"/m/09x0r,/m/0c3f7m"
1,njaBuEnV_LA,490.000,500.000,"/m/09x0r,/m/0c3f7m"
2,3ZrCn3Pujmo,0.000,10.000,/m/0c3f7m
3,cXqSnVsyR9g,30.000,40.000,"/m/01y3hg,/m/0c3f7m"
4,3kK9NzuMc5I,30.000,40.000,"/m/01y3hg,/m/0c3f7m"
5,kOfRBAucXV8,240.000,250.000,"/m/01y3hg,/m/030rvx,/m/09x0r,/m/0c3f7m"
6,Ni79i-5PfA4,500.000,510.000,"/m/01y3hg,/m/09x0r,/m/0c3f7m"
7,YzylenmRMT4,100.000,110.000,"/m/09x0r,/m/0c3f7m"
8,eUw5JADu-0I,420.000,430.000,/m/0c3f7m
9,W_HhkSI07ms,340.000,350.000,"/m/01y3hg,/m/09x0r,/m/0c3f7m,/t/dd00125"


In [30]:
# Download Fire Alarm clips from YouTube

import subprocess
import os
import time
import pandas as pd

fire_alarm_folder = os.path.join(BASE_DIR, "Fire Alarm")
os.makedirs(fire_alarm_folder, exist_ok=True)

results = []

for i, row in fire_alarm_download_candidates.iterrows():

    ytid = row["YTID"]
    start = row["start_seconds"]
    end = row["end_seconds"]

    output_file = os.path.join(
        fire_alarm_folder,
        f"{ytid}_{start}_{end}.wav"
    )

    # If already downloaded, don't download again
    if os.path.exists(output_file):
        print(f"{i+1}/{len(fire_alarm_download_candidates)} ✅ Already exists: {ytid}")

        results.append({
            "YTID": ytid,
            "start_seconds": start,
            "end_seconds": end,
            "label": "Fire Alarm",
            "filename": output_file,
            "status": "already_exists"
        })
        continue

    url = f"https://www.youtube.com/watch?v={ytid}"

    command = [
        "yt-dlp",
        "--no-playlist",
        "--quiet",
        "--no-warnings",
        "--extract-audio",
        "--audio-format", "wav",
        "--download-sections", f"*{start}-{end}",
        "-o", output_file,
        url
    ]

    print(f"{i+1}/{len(fire_alarm_download_candidates)} Downloading: {ytid}")

    result = subprocess.run(
        command,
        capture_output=True,
        text=True
    )

    if result.returncode == 0 and os.path.exists(output_file):
        status = "success"
        print("   ✅ Success")
    else:
        status = "failed"
        print("   ❌ Failed")

    results.append({
        "YTID": ytid,
        "start_seconds": start,
        "end_seconds": end,
        "label": "Fire Alarm",
        "filename": output_file if status == "success" else "",
        "status": status
    })

    time.sleep(1)

# Save download results
fire_alarm_download_results = pd.DataFrame(results)

results_path = os.path.join(
    METADATA_DIR,
    "fire_alarm_download_results.csv"
)

fire_alarm_download_results.to_csv(results_path, index=False)

print("\n==============================")
print("FIRE ALARM DOWNLOAD COMPLETE")
print("==============================")
print("Successful:", (fire_alarm_download_results["status"] == "success").sum())
print("Already existed:", (fire_alarm_download_results["status"] == "already_exists").sum())
print("Failed:", (fire_alarm_download_results["status"] == "failed").sum())
print("Results saved to:", results_path)

1/260 Downloading: r0bhxdy3Ovk
   ❌ Failed
2/260 Downloading: njaBuEnV_LA
   ✅ Success
3/260 Downloading: 3ZrCn3Pujmo
   ✅ Success
4/260 Downloading: cXqSnVsyR9g
   ✅ Success
5/260 Downloading: 3kK9NzuMc5I
   ✅ Success
6/260 Downloading: kOfRBAucXV8
   ✅ Success
7/260 Downloading: Ni79i-5PfA4
   ✅ Success
8/260 Downloading: YzylenmRMT4
   ❌ Failed
9/260 Downloading: eUw5JADu-0I
   ❌ Failed
10/260 Downloading: W_HhkSI07ms
   ✅ Success
11/260 Downloading: 3fJxf2rM990
   ❌ Failed
12/260 Downloading: JU7PS1d2Kic
   ✅ Success
13/260 Downloading: lTK9IgErMA8
   ✅ Success
14/260 Downloading: 3kKfwoe57Eg
   ✅ Success
15/260 Downloading: MaMfDJzAsfc
   ✅ Success
16/260 Downloading: RSo_EFbrVtw
   ✅ Success
17/260 Downloading: Gq0CHblLoqQ
   ❌ Failed
18/260 Downloading: Q2G7rwCx5sc
   ✅ Success
19/260 Downloading: jB1xSB8MojQ
   ✅ Success
20/260 Downloading: c7rAitgz6js
   ✅ Success
21/260 Downloading: n58oIcI9WUU
   ✅ Success
22/260 Downloading: cGWHsTypfP0
   ✅ Success
23/260 Downloading: kTqM

In [31]:
# Get Smoke Alarm candidates from AudioSet unbalanced metadata

smoke_alarm_id = label_ids["Smoke Alarm"]

smoke_alarm_rows = []

with open(unbalanced_path, "r", encoding="utf-8") as f:
    reader = csv.reader(f, skipinitialspace=True)

    for row in reader:
        if not row or row[0].startswith("#"):
            continue

        labels = row[3].split(",")

        if smoke_alarm_id in labels:
            smoke_alarm_rows.append(row)

smoke_alarm_candidates = pd.DataFrame(
    smoke_alarm_rows,
    columns=["YTID", "start_seconds", "end_seconds", "positive_labels"]
)

smoke_alarm_candidates = smoke_alarm_candidates.drop_duplicates(
    subset=["YTID"]
).reset_index(drop=True)

print("Unique Smoke Alarm candidates:", len(smoke_alarm_candidates))
display(smoke_alarm_candidates.head(10))

Unique Smoke Alarm candidates: 426


,YTID,start_seconds,end_seconds,positive_labels
0,-EWgUW9lKes,210.000,220.000,"/m/01y3hg,/m/09x0r"
1,-fKLFmdZXiA,20.000,30.000,/m/01y3hg
2,-uS-9D7njq4,40.000,50.000,"/m/01y3hg,/m/09x0r"
3,013tVBuTrkE,23.000,33.000,/m/01y3hg
4,0Cs-KOQF9t0,40.000,50.000,"/m/01y3hg,/m/0c3f7m"
5,0FjMs2FWcbQ,30.000,40.000,"/m/01y3hg,/m/03kmc9"
6,0JFfTcAXT2Y,30.000,40.000,"/m/01y3hg,/m/02fs_r,/m/09x0r"
7,0VeDtKRDy1Q,30.000,40.000,"/m/01y3hg,/m/07pp_mv,/t/dd00125"
8,0ciZGU2_h6M,0.000,10.000,"/m/01y3hg,/t/dd00125"
9,0o0-GQj6Pow,10.000,20.000,"/m/01y3hg,/m/07pp_mv,/m/09x0r"


In [32]:
# Select Smoke Alarm download candidates

SMOKE_ALARM_TARGET = 260

smoke_alarm_download_candidates = smoke_alarm_candidates.sample(
    n=SMOKE_ALARM_TARGET,
    random_state=42
).reset_index(drop=True)

print("Smoke Alarm candidates selected:", len(smoke_alarm_download_candidates))
display(smoke_alarm_download_candidates.head(10))

Smoke Alarm candidates selected: 260


,YTID,start_seconds,end_seconds,positive_labels
0,yux_XsU7UI8,420.000,430.000,"/m/01y3hg,/m/09x0r"
1,8clWo2RKdZI,30.000,40.000,"/m/01y3hg,/m/030rvx,/m/0c3f7m"
2,ONWBTiUy4To,120.000,130.000,"/m/01y3hg,/m/09x0r"
3,3TMdvmGUWTs,30.000,40.000,/m/01y3hg
4,oaMQwPpGuCs,20.000,30.000,"/m/01y3hg,/t/dd00125"
5,n5n-nCMLboQ,20.000,30.000,/m/01y3hg
6,M-WbEEKI7cY,5.000,15.000,"/m/01y3hg,/m/09x0r"
7,LpqJtM4eA9E,20.000,30.000,/m/01y3hg
8,yQIdLf3BEL8,130.000,140.000,"/m/01y3hg,/t/dd00126"
9,M6eLjTo2jqQ,10.000,20.000,/m/01y3hg


In [33]:
# Download Smoke Alarm clips from YouTube

import subprocess
import os
import time
import pandas as pd

smoke_alarm_folder = os.path.join(BASE_DIR, "Smoke Alarm")
os.makedirs(smoke_alarm_folder, exist_ok=True)

results = []

for i, row in smoke_alarm_download_candidates.iterrows():

    ytid = row["YTID"]
    start = row["start_seconds"]
    end = row["end_seconds"]

    output_file = os.path.join(
        smoke_alarm_folder,
        f"{ytid}_{start}_{end}.wav"
    )

    # If already downloaded, don't download again
    if os.path.exists(output_file):
        print(f"{i+1}/{len(smoke_alarm_download_candidates)} ✅ Already exists: {ytid}")

        results.append({
            "YTID": ytid,
            "start_seconds": start,
            "end_seconds": end,
            "label": "Smoke Alarm",
            "filename": output_file,
            "status": "already_exists"
        })
        continue

    url = f"https://www.youtube.com/watch?v={ytid}"

    command = [
        "yt-dlp",
        "--no-playlist",
        "--quiet",
        "--no-warnings",
        "--extract-audio",
        "--audio-format", "wav",
        "--download-sections", f"*{start}-{end}",
        "-o", output_file,
        url
    ]

    print(f"{i+1}/{len(smoke_alarm_download_candidates)} Downloading: {ytid}")

    result = subprocess.run(
        command,
        capture_output=True,
        text=True
    )

    if result.returncode == 0 and os.path.exists(output_file):
        status = "success"
        print("   ✅ Success")
    else:
        status = "failed"
        print("   ❌ Failed")

    results.append({
        "YTID": ytid,
        "start_seconds": start,
        "end_seconds": end,
        "label": "Smoke Alarm",
        "filename": output_file if status == "success" else "",
        "status": status
    })

    time.sleep(1)

# Save download results
smoke_alarm_download_results = pd.DataFrame(results)

results_path = os.path.join(
    METADATA_DIR,
    "smoke_alarm_download_results.csv"
)

smoke_alarm_download_results.to_csv(results_path, index=False)

print("\n==============================")
print("SMOKE ALARM DOWNLOAD COMPLETE")
print("==============================")
print("Successful:", (smoke_alarm_download_results["status"] == "success").sum())
print("Already existed:", (smoke_alarm_download_results["status"] == "already_exists").sum())
print("Failed:", (smoke_alarm_download_results["status"] == "failed").sum())
print("Results saved to:", results_path)

1/260 Downloading: yux_XsU7UI8
   ❌ Failed
2/260 Downloading: 8clWo2RKdZI
   ✅ Success
3/260 Downloading: ONWBTiUy4To
   ❌ Failed
4/260 Downloading: 3TMdvmGUWTs
   ✅ Success
5/260 Downloading: oaMQwPpGuCs
   ✅ Success
6/260 Downloading: n5n-nCMLboQ
   ❌ Failed
7/260 Downloading: M-WbEEKI7cY
   ❌ Failed
8/260 Downloading: LpqJtM4eA9E
   ❌ Failed
9/260 Downloading: yQIdLf3BEL8
   ✅ Success
10/260 Downloading: M6eLjTo2jqQ
   ✅ Success
11/260 Downloading: xV3w0q-EhvQ
   ❌ Failed
12/260 Downloading: 8JYVd2QFFN4
   ✅ Success
13/260 Downloading: RVoFRijuRdo
   ❌ Failed
14/260 Downloading: jS2Ud6bs4Pc
   ✅ Success
15/260 Downloading: 4TId8kDUeHQ
   ✅ Success
16/260 Downloading: JThDvQBA9m8
   ❌ Failed
17/260 Downloading: _uC0E6Wij4U
   ✅ Success
18/260 Downloading: 8QCztHe9QDM
   ❌ Failed
19/260 Downloading: 0o0-GQj6Pow
   ❌ Failed
20/260 Downloading: b7rGRMn-_ss
   ✅ Success
21/260 Downloading: c7DjT-J8Jeo
   ✅ Success
22/260 Downloading: Z3tJdZo4GZE
   ✅ Success
23/260 Downloading: PSelITJ2v